# 面试问题：Agent 工具调用怎样分类错误、幂等重试并用熔断器阻止故障扩散？

        ## 可直接复述的回答主线

        1. 工具错误要先分成瞬时、永久和未知三类；超时、503 可重试，参数校验和权限错误通常不该重试。
2. 有副作用的调用必须复用同一个 idempotency key，否则超时后盲重试可能重复扣款。
3. 重试使用有上限的指数退避，并把每次尝试的错误类别、延迟和最终状态写入事件账本。
4. 熔断器按工具隔离 CLOSED、OPEN、HALF_OPEN 状态；连续失败达到阈值后快速失败，冷却后只放探针。
5. 全局熔断器会把一个故障工具扩散到健康工具，因此状态键至少应包含工具和租户故障域。
6. 生产还需要 deadline、取消传播、抖动、限流、分布式幂等存储、告警和人工补偿流程。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例模拟库存、支付、CRM 和搜索四个真实工具的七次调用。支付第一次已经扣款但响应超时；CRM 返回永久参数错误；搜索连续 503 后熔断，冷却后半开探针恢复；同一时间的库存调用应保持可用。

In [1]:
calls = [{"id": "call-01", "tool": "inventory", "tick": 0, "key": "stock-A", "outcomes": ["timeout", "ok"]}, {"id": "call-02", "tool": "payment", "tick": 0, "key": "pay-order-9", "outcomes": ["timeout_committed", "ok"]}, {"id": "call-03", "tool": "crm", "tick": 1, "key": "crm-ticket-4", "outcomes": ["validation_error"]}, {"id": "call-04", "tool": "search", "tick": 1, "key": "search-red", "outcomes": ["503", "503"]}, {"id": "call-05", "tool": "search", "tick": 2, "key": "search-blue", "outcomes": ["ok"]}, {"id": "call-06", "tool": "inventory", "tick": 2, "key": "stock-B", "outcomes": ["ok"]}, {"id": "call-07", "tool": "search", "tick": 5, "key": "search-green", "outcomes": ["ok"]}]  # 定义七次跨四个工具且包含真实故障语义的调用。
print("教学实验输入：Agent 工具调用时间线")  # 标记下方是可复现的确定性故障脚本。
print("调用       tick  tool        idempotency_key   scripted outcomes")  # 输出输入预览表头。
for call in calls:  # 逐条展示工具、时间、幂等键和预定结果。
    print(f"{call['id']:<10} {call['tick']:>4}  {call['tool']:<11} {call['key']:<17} {call['outcomes']}")  # 输出当前工具调用的业务字段。
print("关键事实：call-02 的 timeout_committed 表示支付已落账，只是响应丢失。")  # 说明普通 timeout 与有副作用超时的区别。

教学实验输入：Agent 工具调用时间线
调用       tick  tool        idempotency_key   scripted outcomes
call-01       0  inventory   stock-A           ['timeout', 'ok']
call-02       0  payment     pay-order-9       ['timeout_committed', 'ok']
call-03       1  crm         crm-ticket-4      ['validation_error']
call-04       1  search      search-red        ['503', '503']
call-05       2  search      search-blue       ['ok']
call-06       2  inventory   stock-B           ['ok']
call-07       5  search      search-green      ['ok']
关键事实：call-02 的 timeout_committed 表示支付已落账，只是响应丢失。


## 2. Baseline / 基线：所有错误都重试，并为每次重试生成新请求键

这个常见实现会重试永久参数错误；支付超时后又换新键，远端无法识别重复请求，于是同一订单产生两次副作用。

In [2]:
def baseline_blind_retry(call, max_attempts=3):  # 模拟不分类错误且不断更换幂等键的朴素执行器。
    attempts = []  # 保存每次尝试的请求键和结果。
    payment_side_effects = 0  # 统计支付工具实际产生的扣款次数。
    for attempt in range(max_attempts):  # 对任何失败最多执行三次。
        request_key = f"{call['key']}-try-{attempt}"  # 错误地为每次尝试生成全新幂等键。
        outcome = call["outcomes"][min(attempt, len(call["outcomes"]) - 1)]  # 读取确定性远端结果脚本。
        if call["tool"] == "payment" and outcome in {"timeout_committed", "ok"}:  # 判断当前支付请求是否已经产生副作用。
            payment_side_effects += 1  # 新请求键使远端把每次调用都当成新扣款。
        attempts.append({"attempt": attempt + 1, "key": request_key, "outcome": outcome})  # 记录当前盲重试事件。
        if outcome == "ok":  # 只有明确成功才停止基线重试。
            break  # 返回成功后结束循环。
    return {"id": call["id"], "tool": call["tool"], "attempts": attempts, "final": attempts[-1]["outcome"], "side_effects": payment_side_effects}  # 返回基线执行轨迹。
baseline_runs = [baseline_blind_retry(call) for call in calls]  # 对同一批七次调用运行朴素基线。
print("Baseline 盲重试结果")  # 标记下表展示重复副作用和无效重试。
print("调用       tool        attempts  final              payment_side_effects")  # 输出基线结果表头。
for run in baseline_runs:  # 逐调用展示尝试次数与最终结果。
    print(f"{run['id']:<10} {run['tool']:<11} {len(run['attempts']):>8}  {run['final']:<17} {run['side_effects']:>20}")  # 输出当前调用的盲重试代价。
baseline_payment = next(run for run in baseline_runs if run["id"] == "call-02")  # 读取支付超时案例。
baseline_crm = next(run for run in baseline_runs if run["id"] == "call-03")  # 读取永久参数错误案例。

Baseline 盲重试结果
调用       tool        attempts  final              payment_side_effects
call-01    inventory          2  ok                                   0
call-02    payment            2  ok                                   2
call-03    crm                3  validation_error                     0
call-04    search             3  503                                  0
call-05    search             1  ok                                   0
call-06    inventory          1  ok                                   0
call-07    search             1  ok                                   0


## 3. 底层实现：错误分类、同键幂等、退避与按工具熔断

下面不调用重试库。`CircuitBreaker` 显式实现三态转换；执行器先检查熔断，再查幂等结果，随后按错误类别决定重试。日志保存每一次中间状态。

In [3]:
def classify_error(outcome):  # 把远端结果映射为可执行的错误类别。
    if outcome in {"timeout", "timeout_committed", "503", "rate_limit"}:  # 识别通常可恢复的瞬时故障。
        return "transient"  # 允许在预算内退避重试。
    if outcome in {"validation_error", "permission_denied"}:  # 识别重试无法改变的永久故障。
        return "permanent"  # 立即失败并把问题返回规划器。
    return "success" if outcome == "ok" else "unknown"  # 成功或未知错误都显式分类。
class CircuitBreaker:  # 手写按工具隔离的 CLOSED、OPEN、HALF_OPEN 熔断器。
    def __init__(self, failure_threshold=2, cooldown_ticks=3):  # 配置连续失败阈值和逻辑时钟冷却期。
        self.failure_threshold = failure_threshold  # 保存开路所需连续失败数。
        self.cooldown_ticks = cooldown_ticks  # 保存 OPEN 到 HALF_OPEN 的冷却时长。
        self.states = {}  # 按工具保存状态、失败数和开路时间。
    def _entry(self, tool):  # 获取或初始化一个工具的独立熔断状态。
        return self.states.setdefault(tool, {"state": "CLOSED", "failures": 0, "opened_at": None})  # 返回当前工具状态记录。
    def allow(self, tool, tick):  # 在调用前判断工具是否允许通过。
        entry = self._entry(tool)  # 读取当前工具熔断状态。
        if entry["state"] == "OPEN" and tick - entry["opened_at"] >= self.cooldown_ticks:  # 检查开路工具是否完成冷却。
            entry["state"] = "HALF_OPEN"  # 允许一条恢复探针进入半开状态。
        return entry["state"] != "OPEN"  # CLOSED 和 HALF_OPEN 可调用，OPEN 快速失败。
    def record_failure(self, tool, tick):  # 记录一次瞬时远端失败。
        entry = self._entry(tool)  # 读取当前工具状态。
        entry["failures"] += 1  # 累加连续瞬时失败数。
        if entry["failures"] >= self.failure_threshold:  # 检查是否达到开路阈值。
            entry["state"] = "OPEN"  # 进入 OPEN 并阻止后续调用。
            entry["opened_at"] = tick  # 记录逻辑开路时刻供冷却判断。
    def record_success(self, tool):  # 记录成功并恢复健康状态。
        entry = self._entry(tool)  # 读取当前工具状态。
        entry.update({"state": "CLOSED", "failures": 0, "opened_at": None})  # 成功探针关闭熔断并清空连续失败。
idempotency_store = {}  # 模拟生产中需要持久化的幂等结果存储。
payment_effect_count = {}  # 统计每个业务支付键真正执行的副作用次数。
breaker = CircuitBreaker(failure_threshold=2, cooldown_ticks=3)  # 创建按工具隔离的熔断器。
def invoke_remote(call, attempt, request_key):  # 模拟有幂等语义的远端工具。
    if request_key in idempotency_store:  # 检查同一业务请求是否已经完成过。
        return "ok", idempotency_store[request_key], True  # 返回缓存结果且不重复执行副作用。
    outcome = call["outcomes"][min(attempt, len(call["outcomes"]) - 1)]  # 读取当前尝试的确定性脚本结果。
    if call["tool"] == "payment" and outcome == "timeout_committed":  # 模拟支付落账但网络响应丢失。
        payment_effect_count[request_key] = payment_effect_count.get(request_key, 0) + 1  # 对该业务键执行一次真实扣款。
        idempotency_store[request_key] = {"receipt": f"receipt-{call['id']}"}  # 在远端幂等表保存已经完成的结果。
        return outcome, None, False  # 向客户端返回超时以触发同键重试。
    result = {"value": f"result-{call['id']}"} if outcome == "ok" else None  # 为普通成功构造可读结果。
    if outcome == "ok":  # 对成功的无副作用或探针调用保存结果。
        idempotency_store[request_key] = result  # 让后续同键重放可以直接复用。
    return outcome, result, False  # 返回结果、载荷和是否幂等重放。
def execute_with_policy(call, max_attempts=3):  # 执行错误分类、熔断和幂等重试策略。
    ledger = []  # 保存本次调用的逐尝试可观测账本。
    if not breaker.allow(call["tool"], call["tick"]):  # 在访问远端前检查当前工具熔断状态。
        state = breaker._entry(call["tool"])["state"]  # 读取快速失败时的具体状态。
        return {"id": call["id"], "tool": call["tool"], "status": "circuit_open", "attempts": 0, "ledger": ledger, "state": state, "result": None}  # OPEN 时不消耗远端资源。
    for attempt in range(max_attempts):  # 在预算内执行最多三次同键尝试。
        outcome, result, replayed = invoke_remote(call, attempt, call["key"])  # 始终复用业务幂等键调用远端。
        category = classify_error(outcome)  # 把具体错误转为策略类别。
        backoff_ms = 100 * (2 ** attempt) if category == "transient" and attempt + 1 < max_attempts else 0  # 计算有上限的指数退避时间。
        ledger.append({"attempt": attempt + 1, "key": call["key"], "outcome": outcome, "category": category, "backoff_ms": backoff_ms, "replayed": replayed})  # 保存当前尝试全部中间量。
        if category == "success":  # 成功或幂等重放后结束调用。
            breaker.record_success(call["tool"])  # 成功关闭当前工具可能存在的半开状态。
            return {"id": call["id"], "tool": call["tool"], "status": "success", "attempts": attempt + 1, "ledger": ledger, "state": breaker._entry(call["tool"])["state"], "result": result}  # 返回成功及完整账本。
        if category == "permanent" or category == "unknown":  # 永久和未知错误不进行自动重试。
            return {"id": call["id"], "tool": call["tool"], "status": category, "attempts": attempt + 1, "ledger": ledger, "state": breaker._entry(call["tool"])["state"], "result": None}  # 把错误交回上层处理。
        breaker.record_failure(call["tool"], call["tick"])  # 只让瞬时远端故障进入熔断统计。
    return {"id": call["id"], "tool": call["tool"], "status": "retry_exhausted", "attempts": max_attempts, "ledger": ledger, "state": breaker._entry(call["tool"])["state"], "result": None}  # 重试耗尽后返回结构化失败。
policy_runs = [execute_with_policy(call) for call in calls]  # 按逻辑时间顺序执行七次工具调用。
first_payment_run = next(run for run in policy_runs if run["id"] == "call-02")  # 读取支付同键重放轨迹。
search_failure_run = next(run for run in policy_runs if run["id"] == "call-04")  # 读取搜索连续失败开路轨迹。
print("call-02 支付中间账本：", first_payment_run["ledger"])  # 展示超时落账后同键重放且不重复扣款。
print("call-04 搜索中间账本：", search_failure_run["ledger"], "state=", search_failure_run["state"])  # 展示连续 503 到 OPEN 的过程。

call-02 支付中间账本： [{'attempt': 1, 'key': 'pay-order-9', 'outcome': 'timeout_committed', 'category': 'transient', 'backoff_ms': 100, 'replayed': False}, {'attempt': 2, 'key': 'pay-order-9', 'outcome': 'ok', 'category': 'success', 'backoff_ms': 0, 'replayed': True}]
call-04 搜索中间账本： [{'attempt': 1, 'key': 'search-red', 'outcome': '503', 'category': 'transient', 'backoff_ms': 100, 'replayed': False}, {'attempt': 2, 'key': 'search-red', 'outcome': '503', 'category': 'transient', 'backoff_ms': 200, 'replayed': False}, {'attempt': 3, 'key': 'search-red', 'outcome': '503', 'category': 'transient', 'backoff_ms': 0, 'replayed': False}] state= OPEN


## 4. 逐调用结果与结果解读

同一批调用上，CRM 参数错误只执行一次，支付只扣一次，搜索开路期间快速失败，库存不受搜索故障影响；冷却后的搜索探针成功并回到 CLOSED。

In [4]:
baseline_attempts = sum(len(run["attempts"]) for run in baseline_runs)  # 统计朴素基线总远端尝试数。
policy_attempts = sum(run["attempts"] for run in policy_runs)  # 统计策略执行器总远端尝试数。
print("调用       tool        status           attempts  breaker_state  最后一条账本")  # 输出策略结果表头。
for run in policy_runs:  # 逐条展示最终状态和可解释尝试。
    last_event = run["ledger"][-1] if run["ledger"] else None  # OPEN 快速失败没有远端账本。
    print(f"{run['id']:<10} {run['tool']:<11} {run['status']:<16} {run['attempts']:>8}  {run['state']:<13} {last_event}")  # 输出当前调用的完整结果摘要。
corrected_payment_effects = payment_effect_count.get("pay-order-9", 0)  # 读取幂等策略下真实扣款次数。
crm_policy = next(run for run in policy_runs if run["id"] == "call-03")  # 读取参数错误策略结果。
search_open = next(run for run in policy_runs if run["id"] == "call-05")  # 读取冷却前快速失败结果。
healthy_inventory = next(run for run in policy_runs if run["id"] == "call-06")  # 读取搜索开路时库存结果。
recovered_search = next(run for run in policy_runs if run["id"] == "call-07")  # 读取冷却后半开探针结果。
print(f"结果解读：Baseline远端尝试={baseline_attempts}，策略远端尝试={policy_attempts}；支付副作用从{baseline_payment['side_effects']}次降为{corrected_payment_effects}次。")  # 量化错误分类、幂等和快速失败的收益。

调用       tool        status           attempts  breaker_state  最后一条账本
call-01    inventory   success                 2  CLOSED        {'attempt': 2, 'key': 'stock-A', 'outcome': 'ok', 'category': 'success', 'backoff_ms': 0, 'replayed': False}
call-02    payment     success                 2  CLOSED        {'attempt': 2, 'key': 'pay-order-9', 'outcome': 'ok', 'category': 'success', 'backoff_ms': 0, 'replayed': True}
call-03    crm         permanent               1  CLOSED        {'attempt': 1, 'key': 'crm-ticket-4', 'outcome': 'validation_error', 'category': 'permanent', 'backoff_ms': 0, 'replayed': False}
call-04    search      retry_exhausted         3  OPEN          {'attempt': 3, 'key': 'search-red', 'outcome': '503', 'category': 'transient', 'backoff_ms': 0, 'replayed': False}
call-05    search      circuit_open            0  OPEN          None
call-06    inventory   success                 1  CLOSED        {'attempt': 1, 'key': 'stock-B', 'outcome': 'ok', 'category': 'success', 'b

## 5. 失败案例与修正：全局熔断污染健康工具

如果只维护一个全局状态，搜索两次 503 会把库存也判为 OPEN。修正是按工具维护独立故障域；在多租户生产中还可继续细分 endpoint、region 和 tenant。

In [5]:
global_state = {"failures": 0, "open": False}  # 构造错误的单一全局熔断状态。
for outcome in calls[3]["outcomes"]:  # 把搜索调用的两次 503 记到全局计数。
    global_state["failures"] += int(outcome == "503")  # 累加跨工具共享的失败数。
global_state["open"] = global_state["failures"] >= 2  # 达到阈值后错误地把所有工具开路。
global_inventory_allowed = not global_state["open"]  # 全局状态会阻止本应健康的库存调用。
isolated_inventory_allowed = healthy_inventory["status"] == "success"  # 按工具状态允许库存正常成功。
print(f"错误行为：搜索503后 global_open={global_state['open']}，库存允许={global_inventory_allowed}")  # 展示故障跨工具扩散。
print(f"修正行为：search独立状态经历OPEN后恢复，inventory状态={breaker.states['inventory']}，库存允许={isolated_inventory_allowed}")  # 展示故障域隔离效果。

错误行为：搜索503后 global_open=True，库存允许=False
修正行为：search独立状态经历OPEN后恢复，inventory状态={'state': 'CLOSED', 'failures': 0, 'opened_at': None}，库存允许=True


## 6. 生产边界

逻辑 tick 和内存字典只用于教学。线上要使用单调时钟、带随机抖动的 deadline-aware 退避、Redis/数据库幂等记录、跨副本熔断状态、重试预算、取消传播、审计脱敏、支付补偿和人工升级队列。

In [6]:
failure_events = sum(event["category"] != "success" for run in policy_runs for event in run["ledger"])  # 统计策略账本中的失败尝试数。
diagnostics = {"calls": len(calls), "remote_attempts": policy_attempts, "failure_events": failure_events, "circuit_fast_failures": sum(run["status"] == "circuit_open" for run in policy_runs), "duplicate_payment_effects": max(corrected_payment_effects - 1, 0), "search_final_state": breaker.states["search"]["state"]}  # 汇总生产应监控的可靠性和副作用指标。
print("生产监控快照：", diagnostics)  # 输出错误恢复服务的关键指标。

生产监控快照： {'calls': 7, 'remote_attempts': 10, 'failure_events': 6, 'circuit_fast_failures': 1, 'duplicate_payment_effects': 0, 'search_final_state': 'CLOSED'}


## 7. 最小回归测试

断言只验证前面已经打印和解释过的核心行为，不替代案例教学。

In [7]:
assert len(calls) >= 5 and len({call["tool"] for call in calls}) >= 3  # 保证案例包含足够调用和不同故障域。
assert baseline_payment["side_effects"] == 2 and corrected_payment_effects == 1  # 保证重复扣款失败真实复现且同键重放修正有效。
assert len(baseline_crm["attempts"]) == 3 and crm_policy["attempts"] == 1  # 保证永久参数错误不再被无效重试。
assert search_open["status"] == "circuit_open" and search_open["attempts"] == 0  # 保证 OPEN 状态会在访问远端前快速失败。
assert healthy_inventory["status"] == "success" and not global_inventory_allowed and isolated_inventory_allowed  # 保证按工具熔断阻止故障扩散。
assert recovered_search["status"] == "success" and breaker.states["search"]["state"] == "CLOSED"  # 保证冷却后的半开探针能恢复搜索工具。